# 🗽 NYC Congestion Relief Zone (CRZ) Analysis Notebook

***

### 📌 Analytical Overview

| Section | Focus / Objective | Key Methodology & Statistical Tests |
| :--- | :--- | :--- |
| **1. Environment & Setup** | Module imports, global parameters & local CSV ingestion | Local file load (`sample_crz_fixed.csv`), `CRZ_ZONES` configuration |
| **2. EDA & Data Screening** | Data quality audit, schema mapping & pre-filtering | Header mapping, missing value analysis (`Jan 5 – Mar 31`) |
| **3. Policy Impact Modeling** | Quasi-experimental policy evaluation & feature controls | Multi-variable OLS (`M1` → `M3` with HC3 SEs) & Difference-in-Differences (DiD) |
| **4. Core Hypothesis Testing** | Policy impact on earnings, trip volume & service quality | Two-sample $Z$-tests, Welch’s $t$-tests, Proportions $Z$-tests |
| **5. Fare-per-Mile Analyses** | Daily fare density evaluation (Overall & by Service) | 95% $t$-distribution Confidence Intervals & Welch's $t$-tests |
| **6. Spatial & Temporal Variance** | Zone screening & time-of-day driver pay stability | One-Way ANOVA, Tukey HSD Post-Hoc & Two-Way Factorial ANOVA ($\eta_p^2$) |

***

## 1. Environment & Loading File



In [68]:
# Imports & Global Configurations
import getpass
from io import StringIO
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyodbc
from scipy import stats
from sqlalchemy import create_engine
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.weightstats import ztest
from urllib.parse import quote_plus


# Charged Manhattan pickup-zone IDs
CRZ_ZONES = [
    4, 12, 13, 45, 48, 50, 68, 79, 87, 88, 90, 100,
    107, 113, 114, 125, 137, 144, 148, 158, 161, 162,
    163, 164, 170, 186, 209, 211, 224, 229, 230, 231,
    232, 233, 234, 246, 249, 261
]

## 2. Data Screening & Exploratory Data Analysis (EDA)

In [69]:
#  Load Main Analytical Dataset from Local CSV (Header Fix)
csv_file = "sample_crz_fixed.csv"

# 1. Exact column list from SQL Server Object Explorer schema
sql_columns = [
    'operator', 'is_metered', 'pickup_ts', 'dropoff_ts', 'pu_location_id',
    'do_location_id', 'trip_seconds', 'trip_miles', 'passenger_count',
    'passenger_count_reported', 'passenger_paid', 'tip_amount',
    'tip_includes_cash', 'tolls', 'congestion_surcharge', 'cbd_congestion_fee',
    'driver_pay', 'shared_match', 'wav_match', 'hour_band', 'qf_time_invalid',
    'qf_duration_short', 'qf_duration_long', 'qf_out_of_period',
    'qf_zero_distance', 'qf_distance_extreme', 'qf_speed_implausible',
    'qf_negative_money', 'qf_zone_missing', 'qf_zone_unknown',
    'qf_negative_cbd_congestion_fee', 'qf_zero_cbd_congestion_fee',
    'qf_any', 'service', 'year', 'month', 'request_ts', 'zone', 'is_crz'
]

# 2. Read CSV treating row 0 as data
df = pd.read_csv(csv_file, header=None)

# 3. Assign headers if counts match
if len(df.columns) == len(sql_columns):
    df.columns = sql_columns
    print(f"✅ Successfully mapped all {len(sql_columns)} columns!")
else:
    print(f"⚠️ Mismatch: CSV has {len(df.columns)} columns, expected {len(sql_columns)}.")

# 4. Parse datetimes safely
for col in ['pickup_ts', 'dropoff_ts', 'request_ts']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print(f"Loaded {len(df):,} rows.")

/tmp/ipykernel_3352/948628985.py:19: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file, header=None)


✅ Successfully mapped all 39 columns!
Loaded 865,062 rows.


In [70]:
#  Missing Value Audit
na_percent = df.isna().mean() * 100
print("Columns with missing values (%):")
display(na_percent[na_percent > 0].sort_values(ascending=False))

Columns with missing values (%):


,0
passenger_count,93.4745
cbd_congestion_fee,53.6363
wav_match,19.2372
shared_match,19.2372
request_ts,19.2372
driver_pay,19.2372
congestion_surcharge,12.7117
trip_miles,8.2885
tip_amount,8.2885
passenger_paid,8.2885


In [71]:
# Preliminary Data Filtering (Jan 5 – Mar 31)
df['pickup_ts'] = pd.to_datetime(df['pickup_ts'])
df_filtered = df[~((df['pickup_ts'].dt.month == 1) & (df['pickup_ts'].dt.day <= 4))]
df_filtered = df_filtered[df_filtered['qf_any'] == 0]

df_crz = df_filtered[df_filtered['is_crz'] == 1]
df_not_crz = df_filtered[df_filtered['is_crz'] == 0]

## 3. Policy Impact Modeling

In [72]:
# Prepare clean FHVHV regression dataset
df_reg = df_filtered[
    (df_filtered['service'] == 'fhvhv') &
    (df_filtered['driver_pay'].notna()) &
    (df_filtered['trip_miles'] > 0) &
    (df_filtered['trip_seconds'] > 0)
].copy()

# Dynamic Borough Mapping (Manhattan vs. Other)
borough_map = {z: 'Manhattan' for z in CRZ_ZONES}
df_reg['borough'] = df_reg['pu_location_id'].map(borough_map).fillna('Other')
df_reg['post_2025'] = (df_reg['year'] == 2025).astype(int)

# Formulate Nested Regressions
f1 = 'driver_pay ~ is_crz + post_2025 + is_crz:post_2025'
f2 = f1 + ' + trip_miles + trip_seconds'
f3 = f2 + ' + C(hour_band) + C(borough)'

# Fit OLS Models with Heteroscedasticity-Robust Standard Errors (HC3)
m1 = smf.ols(f1, data=df_reg).fit(cov_type='HC3')
m2 = smf.ols(f2, data=df_reg).fit(cov_type='HC3')
m3 = smf.ols(f3, data=df_reg).fit(cov_type='HC3')

# Print Policy Interaction Summary (is_crz:post_2025)
print("=== OLS REGRESSION POLICY IMPACT SUMMARY ===")
print(f"Model 1 (Unadjusted):       ${m1.params['is_crz:post_2025']:+.4f} (p = {m1.pvalues['is_crz:post_2025']:.4f})")
print(f"Model 2 (+ Distance/Time): ${m2.params['is_crz:post_2025']:+.4f} (p = {m2.pvalues['is_crz:post_2025']:.4f})")
print(f"Model 3 (+ Hour/Borough):   ${m3.params['is_crz:post_2025']:+.4f} (p = {m3.pvalues['is_crz:post_2025']:.4f})")

=== OLS REGRESSION POLICY IMPACT SUMMARY ===
Model 1 (Unadjusted):       $-0.0432 (p = 0.6031)
Model 2 (+ Distance/Time): $+0.5534 (p = 0.0000)
Model 3 (+ Hour/Borough):   $+0.5454 (p = 0.0000)


In [73]:
# ==============================================================================
# DIFFERENCE-IN-DIFFERENCES (DiD) INTERACTION REGRESSION
# ==============================================================================

# Prepare dataset with treatment and post-period indicators
df_did = df_filtered[
    (df_filtered['service'] == 'fhvhv') &
    (df_filtered['driver_pay'].notna())
].copy()

df_did['post_2025'] = (df_did['year'] == 2025).astype(int)
df_did['did_interaction'] = df_did['is_crz'] * df_did['post_2025']

# Fit Formal DiD Model
model_did = smf.ols('driver_pay ~ is_crz + post_2025 + is_crz:post_2025', data=df_did).fit()

print("=== DIFFERENCE-IN-DIFFERENCES (DiD) REGRESSION SUMMARY ===")
print(model_did.summary().tables[1])

# Extract key statistics
did_coef = model_did.params['is_crz:post_2025']
did_pvalue = model_did.pvalues['is_crz:post_2025']

print(f"\nNet Policy Impact (DiD Estimate): ${did_coef:+.4f} per trip")
print(f"P-value:                        {did_pvalue:.4f}")
print(f"Decision:                       {'Statistically Significant Net Effect' if did_pvalue < 0.05 else 'No Statistically Significant Net Effect'}")

=== DIFFERENCE-IN-DIFFERENCES (DiD) REGRESSION SUMMARY ===
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           16.0025      0.025    634.514      0.000      15.953      16.052
is_crz               5.7236      0.054    106.145      0.000       5.618       5.829
post_2025            0.1883      0.036      5.285      0.000       0.118       0.258
is_crz:post_2025    -0.0432      0.076     -0.567      0.571      -0.192       0.106

Net Policy Impact (DiD Estimate): $-0.0432 per trip
P-value:                        0.5707
Decision:                       No Statistically Significant Net Effect


## 4. Core Hypothesis Testing & Service Metrics

In [74]:
# ==============================================================================
# DRIVER EARNINGS ASSESSMENT (FHVHV)
# ==============================================================================

df_crz_fh = df_crz[df_crz['service'] == 'fhvhv']
pay_2024_crz = df_crz_fh[df_crz_fh['year'] == 2024]['driver_pay']
pay_2025_crz = df_crz_fh[df_crz_fh['year'] == 2025]['driver_pay']

z_stat_pay_crz, p_val_pay_crz = ztest(pay_2025_crz, pay_2024_crz, alternative='smaller')

print("=== CRZ DRIVER EARNINGS TEST ===")
print(f"Average Pay 2024: ${pay_2024_crz.mean():.2f} (n={len(pay_2024_crz):,})")
print(f"Average Pay 2025: ${pay_2025_crz.mean():.2f} (n={len(pay_2025_crz):,})")
print(f"Z-statistic:   {z_stat_pay_crz:.4f}")
print(f"P-value:       {p_val_pay_crz:.4f}")
print(f"Decision:      {'Reject H0' if p_val_pay_crz < 0.05 else 'Fail to Reject H0'}\n")

# 1B. Control Group Driver Earnings Test (Outside CRZ)
df_not_crz_fh = df_not_crz[df_not_crz['service'] == 'fhvhv']
pay_2024_ctrl = df_not_crz_fh[df_not_crz_fh['year'] == 2024]['driver_pay']
pay_2025_ctrl = df_not_crz_fh[df_not_crz_fh['year'] == 2025]['driver_pay']

z_stat_pay_ctrl, p_val_pay_ctrl = ztest(pay_2025_ctrl, pay_2024_ctrl, alternative='smaller')

print("=== [CONTROL GROUP] OUTSIDE CRZ DRIVER EARNINGS ===")
print(f"Average Pay 2024: ${pay_2024_ctrl.mean():.2f} (n={len(pay_2024_ctrl):,})")
print(f"Average Pay 2025: ${pay_2025_ctrl.mean():.2f} (n={len(pay_2025_ctrl):,})")
print(f"Z-statistic:   {z_stat_pay_ctrl:.4f}")
print(f"P-value:       {p_val_pay_ctrl:.4f}")
print(f"Decision:      {'Reject H0' if p_val_pay_ctrl < 0.05 else 'Fail to Reject H0'}")

=== CRZ DRIVER EARNINGS TEST ===
Average Pay 2024: $21.73 (n=73,743)
Average Pay 2025: $21.87 (n=74,358)
Z-statistic:   1.9201
P-value:       0.9726
Decision:      Fail to Reject H0

=== [CONTROL GROUP] OUTSIDE CRZ DRIVER EARNINGS ===
Average Pay 2024: $16.00 (n=263,368)
Average Pay 2025: $16.19 (n=264,226)
Z-statistic:   5.4918
P-value:       1.0000
Decision:      Fail to Reject H0


In [75]:
# ==============================================================================
#  TRIP VOLUME ASSESSMENT
# ==============================================================================
df_crz_daily = df_crz.groupby(['year', df_crz['pickup_ts'].dt.date]).size().reset_index(name='daily_trip_volume')
vol_crz_2024 = df_crz_daily[df_crz_daily['year'] == 2024]['daily_trip_volume']
vol_crz_2025 = df_crz_daily[df_crz_daily['year'] == 2025]['daily_trip_volume']

t_stat_vol_crz, p_val_vol_crz = stats.ttest_ind(vol_crz_2025, vol_crz_2024, equal_var=False, alternative='less')

print("=== CRZ TRIP VOLUME TEST ===")
print(f"Average Daily Trip Volume 2024: {vol_crz_2024.mean():.2f} (n={len(vol_crz_2024)})")
print(f"Average Daily Trip Volume 2025: {vol_crz_2025.mean():.2f} (n={len(vol_crz_2025)})")
print(f"T-statistic:                    {t_stat_vol_crz:.4f}")
print(f"P-value:                        {p_val_vol_crz:.4f}")
print(f"Decision:                       {'Reject H0' if p_val_vol_crz < 0.05 else 'Fail to Reject H0'}\n")

# 2B. Control Group Daily Trip Volume Test
df_not_crz_daily = df_not_crz.groupby(['year', df_not_crz['pickup_ts'].dt.date]).size().reset_index(name='daily_trip_volume')
vol_ctrl_2024 = df_not_crz_daily[df_not_crz_daily['year'] == 2024]['daily_trip_volume']
vol_ctrl_2025 = df_not_crz_daily[df_not_crz_daily['year'] == 2025]['daily_trip_volume']

t_stat_vol_ctrl, p_val_vol_ctrl = stats.ttest_ind(vol_ctrl_2025, vol_ctrl_2024, equal_var=False, alternative='less')

print("=== [CONTROL GROUP] OUTSIDE CRZ TRIP VOLUME ===")
print(f"Average Daily Trip Volume 2024: {vol_ctrl_2024.mean():.2f} (n={len(vol_ctrl_2024)})")
print(f"Average Daily Trip Volume 2025: {vol_ctrl_2025.mean():.2f} (n={len(vol_ctrl_2025)})")
print(f"T-statistic:                    {t_stat_vol_ctrl:.4f}")
print(f"P-value:                        {p_val_vol_ctrl:.4f}")
print(f"Decision:                       {'Reject H0' if p_val_vol_ctrl < 0.05 else 'Fail to Reject H0'}")

=== CRZ TRIP VOLUME TEST ===
Average Daily Trip Volume 2024: 1078.28 (n=88)
Average Daily Trip Volume 2025: 1108.59 (n=87)
T-statistic:                    1.0429
P-value:                        0.8507
Decision:                       Fail to Reject H0

=== [CONTROL GROUP] OUTSIDE CRZ TRIP VOLUME ===
Average Daily Trip Volume 2024: 3646.69 (n=88)
Average Daily Trip Volume 2025: 3731.41 (n=87)
T-statistic:                    1.3061
P-value:                        0.9034
Decision:                       Fail to Reject H0


In [76]:
# ==============================================================================
#  SHORT TRIP RESILIENCE (< 2 MILES)
# ==============================================================================
# 3A. CRZ Short Trip Proportion Test
df_2024_crz, df_2025_crz = df_crz[df_crz['year'] == 2024], df_crz[df_crz['year'] == 2025]
cnt_crz_24, nobs_crz_24 = (df_2024_crz['trip_miles'] < 2.0).sum(), len(df_2024_crz)
cnt_crz_25, nobs_crz_25 = (df_2025_crz['trip_miles'] < 2.0).sum(), len(df_2025_crz)

z_stat_short_crz, p_val_short_crz = proportions_ztest(count=[cnt_crz_25, cnt_crz_24], nobs=[nobs_crz_25, nobs_crz_24], alternative='smaller')

print("=== CRZ SHORT TRIP PROPORTION TEST ===")
print(f"2024: {cnt_crz_24:,} / {nobs_crz_24:,} ({cnt_crz_24 / nobs_crz_24:.2%})")
print(f"2025: {cnt_crz_25:,} / {nobs_crz_25:,} ({cnt_crz_25 / nobs_crz_25:.2%})")
print(f"Z-statistic:          {z_stat_short_crz:.4f}")
print(f"P-value (one-tailed): {p_val_short_crz:.4f}")
print(f"Decision:             {'Reject H0' if p_val_short_crz < 0.05 else 'Fail to Reject H0'}\n")

# 3B. Control Group Short Trip Proportion Test
df_2024_ctrl, df_2025_ctrl = df_not_crz[df_not_crz['year'] == 2024], df_not_crz[df_not_crz['year'] == 2025]
cnt_ctrl_24, nobs_ctrl_24 = (df_2024_ctrl['trip_miles'] < 2.0).sum(), len(df_2024_ctrl)
cnt_ctrl_25, nobs_ctrl_25 = (df_2025_ctrl['trip_miles'] < 2.0).sum(), len(df_2025_ctrl)

z_stat_short_ctrl, p_val_short_ctrl = proportions_ztest(count=[cnt_ctrl_25, cnt_ctrl_24], nobs=[nobs_ctrl_25, nobs_ctrl_24], alternative='smaller')

print("=== [CONTROL GROUP] OUTSIDE CRZ SHORT TRIP PROPORTION ===")
print(f"2024: {cnt_ctrl_24:,} / {nobs_ctrl_24:,} ({cnt_ctrl_24 / nobs_ctrl_24:.2%})")
print(f"2025: {cnt_ctrl_25:,} / {nobs_ctrl_25:,} ({cnt_ctrl_25 / nobs_ctrl_25:.2%})")
print(f"Z-statistic:          {z_stat_short_ctrl:.4f}")
print(f"P-value (one-tailed): {p_val_short_ctrl:.4f}")
print(f"Decision:             {'Reject H0' if p_val_short_ctrl < 0.05 else 'Fail to Reject H0'}")

=== CRZ SHORT TRIP PROPORTION TEST ===
2024: 27,117 / 94,889 (28.58%)
2025: 27,980 / 96,447 (29.01%)
Z-statistic:          2.0921
P-value (one-tailed): 0.9818
Decision:             Fail to Reject H0

=== [CONTROL GROUP] OUTSIDE CRZ SHORT TRIP PROPORTION ===
2024: 105,823 / 320,909 (32.98%)
2025: 108,128 / 324,633 (33.31%)
Z-statistic:          2.8312
P-value (one-tailed): 0.9977
Decision:             Fail to Reject H0


In [77]:
# ==============================================================================
#  SERVICE QUALITY: RIDER WAIT TIME (FHVHV)
# ==============================================================================

df_crz_wait = df_crz[df_crz['service'] == 'fhvhv'].copy()
df_crz_wait['request_ts'] = pd.to_datetime(df_crz_wait['request_ts'])
df_crz_wait['wait_time_mins'] = (df_crz_wait['pickup_ts'] - df_crz_wait['request_ts']).dt.total_seconds() / 60.0

wait_crz_2024 = df_crz_wait[df_crz_wait['year'] == 2024]['wait_time_mins']
wait_crz_2025 = df_crz_wait[df_crz_wait['year'] == 2025]['wait_time_mins']

z_stat_wait_crz, p_val_wait_crz = ztest(wait_crz_2025, wait_crz_2024, alternative='larger')

print("=== CRZ RIDER WAIT TIME TEST ===")
print(f"Average Wait Time 2024: {wait_crz_2024.mean():.2f} mins (n={len(wait_crz_2024):,})")
print(f"Average Wait Time 2025: {wait_crz_2025.mean():.2f} mins (n={len(wait_crz_2025):,})")
print(f"Z-statistic:         {z_stat_wait_crz:.4f}")
print(f"P-value (one-tailed):{p_val_wait_crz:.4f}")
print(f"Decision:            {'Reject H0' if p_val_wait_crz < 0.05 else 'Fail to Reject H0'}\n")

# 4B. Control Group Rider Wait Time Test
df_ctrl_wait = df_not_crz[df_not_crz['service'] == 'fhvhv'].copy()
df_ctrl_wait['request_ts'] = pd.to_datetime(df_ctrl_wait['request_ts'])
df_ctrl_wait['wait_time_mins'] = (df_ctrl_wait['pickup_ts'] - df_ctrl_wait['request_ts']).dt.total_seconds() / 60.0

wait_ctrl_2024 = df_ctrl_wait[df_ctrl_wait['year'] == 2024]['wait_time_mins']
wait_ctrl_2025 = df_ctrl_wait[df_ctrl_wait['year'] == 2025]['wait_time_mins']

z_stat_wait_ctrl, p_val_wait_ctrl = ztest(wait_ctrl_2025, wait_ctrl_2024, alternative='larger')

print("=== [CONTROL GROUP] NON-CRZ RIDER WAIT TIME ===")
print(f"Mean Wait Time 2024: {wait_ctrl_2024.mean():.2f} mins (n={len(wait_ctrl_2024):,})")
print(f"Mean Wait Time 2025: {wait_ctrl_2025.mean():.2f} mins (n={len(wait_ctrl_2025):,})")
print(f"Z-statistic:         {z_stat_wait_ctrl:.4f}")
print(f"P-value (one-tailed):{p_val_wait_ctrl:.4f}")
print(f"Decision:            {'Reject H0' if p_val_wait_ctrl < 0.05 else 'Fail to Reject H0'}")

=== CRZ RIDER WAIT TIME TEST ===
Average Wait Time 2024: 4.12 mins (n=73,743)
Average Wait Time 2025: 4.41 mins (n=74,358)
Z-statistic:         16.7658
P-value (one-tailed):0.0000
Decision:            Reject H0

=== [CONTROL GROUP] NON-CRZ RIDER WAIT TIME ===
Mean Wait Time 2024: 4.60 mins (n=263,368)
Mean Wait Time 2025: 4.99 mins (n=264,226)
Z-statistic:         40.2225
P-value (one-tailed):0.0000
Decision:            Reject H0


In [78]:
# ==============================================================================
#  SERVICE QUALITY: SHARED-RIDE MATCH RATE (FHVHV)
# ==============================================================================

df_crz_shared = df_crz[df_crz['service'] == 'fhvhv'].copy()
df_crz_shared['is_matched'] = df_crz_shared['shared_match'].astype(int)

df_crz_24, df_crz_25 = df_crz_shared[df_crz_shared['year'] == 2024], df_crz_shared[df_crz_shared['year'] == 2025]
cnt_match_crz_24, nobs_match_crz_24 = df_crz_24['is_matched'].sum(), len(df_crz_24)
cnt_match_crz_25, nobs_match_crz_25 = df_crz_25['is_matched'].sum(), len(df_crz_25)

z_stat_match_crz, p_val_match_crz = proportions_ztest(count=[cnt_match_crz_25, cnt_match_crz_24], nobs=[nobs_match_crz_25, nobs_match_crz_24], alternative='smaller')

print("=== CRZ SHARED-RIDE MATCH RATE TEST ===")
print(f"2024: {cnt_match_crz_24:,} / {nobs_match_crz_24:,} ({cnt_match_crz_24 / nobs_match_crz_24:.2%})")
print(f"2025: {cnt_match_crz_25:,} / {nobs_match_crz_25:,} ({cnt_match_crz_25 / nobs_match_crz_25:.2%})")
print(f"Z-statistic:          {z_stat_match_crz:.4f}")
print(f"P-value (one-tailed): {p_val_match_crz:.4f}")
print(f"Decision:             {'Reject H0' if p_val_match_crz < 0.05 else 'Fail to Reject H0'}\n")

# 5B. Control Group Shared-Ride Match Rate Test
df_ctrl_shared = df_not_crz[df_not_crz['service'] == 'fhvhv'].copy()
df_ctrl_shared['is_matched'] = df_ctrl_shared['shared_match'].astype(int)

df_ctrl_24, df_ctrl_25 = df_ctrl_shared[df_ctrl_shared['year'] == 2024], df_ctrl_shared[df_ctrl_shared['year'] == 2025]
cnt_match_ctrl_24, nobs_match_ctrl_24 = df_ctrl_24['is_matched'].sum(), len(df_ctrl_24)
cnt_match_ctrl_25, nobs_match_ctrl_25 = df_ctrl_25['is_matched'].sum(), len(df_ctrl_25)

z_stat_match_ctrl, p_val_match_ctrl = proportions_ztest(count=[cnt_match_ctrl_25, cnt_match_ctrl_24], nobs=[nobs_match_ctrl_25, nobs_match_ctrl_24], alternative='smaller')

print("=== [CONTROL GROUP] NON-CRZ SHARED-RIDE MATCH RATE ===")
print(f"2024: {cnt_match_ctrl_24:,} / {nobs_match_ctrl_24:,} ({cnt_match_ctrl_24 / nobs_match_ctrl_24:.2%})")
print(f"2025: {cnt_match_ctrl_25:,} / {nobs_match_ctrl_25:,} ({cnt_match_ctrl_25 / nobs_match_ctrl_25:.2%})")
print(f"Z-statistic:          {z_stat_match_ctrl:.4f}")
print(f"P-value (one-tailed): {p_val_match_ctrl:.4f}")
print(f"Decision:             {'Reject H0' if p_val_match_ctrl < 0.05 else 'Fail to Reject H0'}")

=== CRZ SHARED-RIDE MATCH RATE TEST ===
2024: 985 / 73,743 (1.34%)
2025: 952 / 74,358 (1.28%)
Z-statistic:          -0.9387
P-value (one-tailed): 0.1739
Decision:             Fail to Reject H0

=== [CONTROL GROUP] NON-CRZ SHARED-RIDE MATCH RATE ===
2024: 1,852 / 263,368 (0.70%)
2025: 4,033 / 264,226 (1.53%)
Z-statistic:          28.4648
P-value (one-tailed): 1.0000
Decision:             Fail to Reject H0


## 5. Daily Fare-per-Mile Analyses

###  Overall 95% Confidence Interval for Mean Fare per Mile

In [79]:

df_clean = df[(df['qf_any'] == 0) & (df['trip_miles'] > 0) & (df['passenger_paid'] > 0)].copy()

# Filter date windows: Jan 5 – Mar 31 for 2024 and 2025
df_clean['trip_date'] = df_clean['pickup_ts'].dt.date
mask_2024 = (df_clean['pickup_ts'] >= '2024-01-05') & (df_clean['pickup_ts'] < '2024-04-01')
mask_2025 = (df_clean['pickup_ts'] >= '2025-01-05') & (df_clean['pickup_ts'] < '2025-04-01')

df_3b1_raw = df_clean[mask_2024 | mask_2025].copy()

# Daily aggregation for fare per mile
df_3b1 = df_3b1_raw.groupby(['year', 'trip_date']).agg(
    total_fare=('passenger_paid', 'sum'),
    total_miles=('trip_miles', 'sum')
).reset_index()

df_3b1["fare_per_mile"] = df_3b1["total_fare"] / df_3b1["total_miles"]

# 95% Confidence Interval Calculation
results_3b1 = []
for yr in [2024, 2025]:
    x = df_3b1.loc[df_3b1["year"] == yr, "fare_per_mile"].dropna()
    n = len(x)
    mean = x.mean()
    se = x.std(ddof=1) / np.sqrt(n)
    margin = stats.t.ppf(0.975, n - 1) * se  # Fixed missing '*' operator
    results_3b1.append({
        "year": yr,
        "n_days": n,
        "mean_fare_per_mile": mean,
        "lower_95_CI": mean - margin,
        "upper_95_CI": mean + margin
    })

display(pd.DataFrame(results_3b1).round(4))

,year,n_days,mean_fare_per_mile,lower_95_CI,upper_95_CI
0,2024,87,5.6867,5.6420,5.7314
1,2025,86,5.8763,5.8118,5.9408


### Service-by-Service Fare-per-Mile Impact Comparison

In [80]:

mask_base = (
    (df['qf_any'] == 0) &
    (df['trip_miles'] > 0) &
    (df['passenger_paid'] > 0) &
    (df['is_crz'] == 1) &
    (df['service'].isin(['yellow', 'green', 'fhvhv', 'fhv']))
)
df_clean_3b2 = df[mask_base].copy()

# Filter date windows: Jan 5 – Mar 31 for 2024 and 2025
df_clean_3b2['trip_date'] = df_clean_3b2['pickup_ts'].dt.date
mask_2024 = (df_clean_3b2['pickup_ts'] >= '2024-01-05') & (df_clean_3b2['pickup_ts'] < '2024-04-01')
mask_2025 = (df_clean_3b2['pickup_ts'] >= '2025-01-05') & (df_clean_3b2['pickup_ts'] < '2025-04-01')

df_3b2 = df_clean_3b2[mask_2024 | mask_2025].copy()

# Daily aggregation by service, year, and trip_date
daily_3b2 = df_3b2.groupby(["service", "year", "trip_date"]).agg(
    total_fare=("passenger_paid", "sum"),
    total_miles=("trip_miles", "sum")
).reset_index()

daily_3b2["fare_per_mile"] = daily_3b2["total_fare"] / daily_3b2["total_miles"]

# Execute Welch's t-Test for each service type
results_3b2 = []
for srv in ["yellow", "green", "fhvhv", "fhv"]:
    s_data = daily_3b2[daily_3b2["service"] == srv]
    f24 = s_data.loc[s_data["year"] == 2024, "fare_per_mile"].dropna()
    f25 = s_data.loc[s_data["year"] == 2025, "fare_per_mile"].dropna()

    if len(f24) == 0 or len(f25) == 0:
        continue

    t_stat, p_val = stats.ttest_ind(f25, f24, equal_var=False)
    m24, m25 = f24.mean(), f25.mean()
    pct_change = ((m25 - m24) / m24) * 100  # Fixed broken subtraction & multiplication syntax

    results_3b2.append({
        "service": srv,
        "mean_2024": m24,
        "mean_2025": m25,
        "pct_change": pct_change,
        "t_stat": t_stat,
        "p_value": p_val,
        "decision": "Reject H0" if p_val < 0.05 else "Fail to reject H0"
    })

display(pd.DataFrame(results_3b2).round(4))

,service,mean_2024,mean_2025,pct_change,t_stat,p_value,decision
0,yellow,7.2710,7.1502,-1.6608,-1.4531,0.1480,Fail to reject H0
1,green,7.1099,7.6410,7.4700,1.1358,0.2597,Fail to reject H0
2,fhvhv,6.5298,7.1624,9.6886,11.2250,0.0000,Reject H0


## 6.  Spatial Variance & High-Depress Zone Screening

In [81]:
# ==============================================================================
# Driver Pay Across Charged Zones (ANOVA & Tukey HSD)
# ==============================================================================
mask_3c1 = (
    (df['service'] == 'fhvhv') &
    (df['qf_any'] == 0) &
    (df['driver_pay'].notna()) &
    (df['pu_location_id'].isin(CRZ_ZONES)) &
    (df['pickup_ts'] >= '2025-01-05') &
    (df['pickup_ts'] < '2025-04-01')
)

df_3c1 = df[mask_3c1][['pu_location_id', 'driver_pay']].copy()
df_3c1['driver_pay'] = df_3c1['driver_pay'].astype(float)

# Execute One-Way ANOVA across CRZ pickup locations
anova_groups = [g["driver_pay"].dropna().values for _, g in df_3c1.groupby("pu_location_id")]
anova_f, anova_p = stats.f_oneway(*anova_groups)
print(f"One-way ANOVA F-statistic: {anova_f:.4f}, p-value: {anova_p:.6g}")

# Run Tukey HSD Post-Hoc Pairwise Test
tukey = pairwise_tukeyhsd(endog=df_3c1["driver_pay"], groups=df_3c1["pu_location_id"], alpha=0.05)
tukey_df = pd.DataFrame(data=tukey._results_table.data[1:], columns=tukey._results_table.data[0])

# Aggregate zone-level earnings summary
zone_summary = df_3c1.groupby("pu_location_id")["driver_pay"].agg(
    n="count",
    mean_driver_pay="mean",
    median_driver_pay="median",
    sd_driver_pay="std"
).reset_index()

# Map significantly lower earning zone pairs
zone_means = zone_summary.set_index("pu_location_id")["mean_driver_pay"].to_dict()
flag_results = []
for z in zone_means:
    lower_cnt = 0
    z_pairs = tukey_df[(tukey_df["group1"] == z) | (tukey_df["group2"] == z)]
    for _, row in z_pairs.iterrows():
        if bool(row["reject"]):
            other = row["group2"] if row["group1"] == z else row["group1"]
            if zone_means[z] < zone_means[other]:
                lower_cnt += 1
    flag_results.append({"pu_location_id": z, "significantly_lower_than_n_zones": lower_cnt})

# Final depressed zone ranking table
final_zone_table = zone_summary.merge(pd.DataFrame(flag_results), on="pu_location_id").sort_values(
    "mean_driver_pay", ascending=True
)

print("\nTop Low-Earning Pickup Zones:")
display(final_zone_table.head(10))

One-way ANOVA F-statistic: 28.4606, p-value: 8.66534e-195


/usr/local/lib/python3.13/dist-packages/scipy/integrate/_quadpack_py.py:1264: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  quad_r = quad(f, low, high, args=args, full_output=self.full_output,



Top Low-Earning Pickup Zones:


,pu_location_id,n,mean_driver_pay,median_driver_pay,sd_driver_pay,significantly_lower_than_n_zones
14,114,1374,16.5406,13.6300,10.3506,19
7,79,1373,16.6433,13.1900,11.6562,19
24,170,1371,16.9765,13.5900,12.1971,18
0,4,1373,17.2891,14.5000,11.4531,17
12,107,1371,17.4342,13.3100,13.2470,15
36,249,1372,17.4562,13.3950,12.5068,15
13,113,1372,17.5930,13.5750,13.7307,14
17,144,1378,17.6078,14.5500,11.5501,14
33,233,1371,17.6505,13.6000,12.8256,14
19,158,1367,17.6552,13.7300,12.5848,14


In [85]:
# ==============================================================================
# TWO-WAY ANOVA WITH EFFECT SIZE DIAGNOSTICS (ETA-SQUARED)
# ==============================================================================
df_anova = df_crz[
    (df_crz['service'] == 'fhvhv') &
    (df_crz['driver_pay'].notna()) &
    (df_crz['hour_band'].notna())
].copy()

# 1. Fit 2-Way Factorial ANOVA Model
model_2way = smf.ols('driver_pay ~ C(year) * C(hour_band)', data=df_anova).fit()
anova_table = sm.stats.anova_lm(model_2way, typ=2)

# 2. Compute Partial Eta-Squared (eta_p^2) to measure practical effect size
anova_table['eta_sq_perc'] = (anova_table['sum_sq'] / (anova_table['sum_sq'] + anova_table.loc['Residual', 'sum_sq'])) * 100

print("=== TWO-WAY ANOVA TABLE WITH EFFECT SIZES ===")
display(anova_table.round(4))

# Print interpretation of practical effect size
interaction_eta = anova_table.loc['C(year):C(hour_band)', 'eta_sq_perc']
print(f"\nPractical Significance Note:")
print(f"Year x Hour Band interaction explains only {interaction_eta:.2f}% of driver pay variance.")
print("Takeaway: Although p < 0.05 due to large sample scale, the effect size is negligible, supporting time-of-day stability.")

# 3. Hour Band Pay Breakdown
pay_by_hour = df_anova.groupby(['hour_band', 'year'])['driver_pay'].mean().unstack()
pay_by_hour['Abs_Change'] = pay_by_hour[2025] - pay_by_hour[2024]
pay_by_hour['Pct_Change'] = (pay_by_hour['Abs_Change'] / pay_by_hour[2024]) * 100

print("\n=== DRIVER PAY BREAKDOWN BY HOUR BAND ===")
display(pay_by_hour.round(2))

=== TWO-WAY ANOVA TABLE WITH EFFECT SIZES ===


,sum_sq,df,F,PR(>F),eta_sq_perc
C(year),725.8782,1.0000,3.4419,0.0636,0.0023
C(hour_band),"106,857.4079",4.0000,126.6727,0.0000,0.3410
C(year):C(hour_band),"6,867.9264",4.0000,8.1415,0.0000,0.0220
Residual,"31,231,323.1705","148,091.0000",NaN,NaN,50.0000



Practical Significance Note:
Year x Hour Band interaction explains only 0.02% of driver pay variance.
Takeaway: Although p < 0.05 due to large sample scale, the effect size is negligible, supporting time-of-day stability.

=== DRIVER PAY BREAKDOWN BY HOUR BAND ===


year,2024,2025,Abs_Change,Pct_Change
hour_band,,,,
1_overnight,21.2800,22.1100,0.8300,3.9000
2_am_peak,22.2100,22.5500,0.3400,1.5200
3_midday,22.9500,22.6100,-0.3300,-1.4600
4_pm_peak,21.9100,21.6000,-0.3200,-1.4400
5_evening,20.1900,20.3800,0.1900,0.9400
